In [11]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch.nn import functional as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from domain_shift.core.config import settings
from domain_shift.CycleGAN.data_loader import get_data_loader
from domain_shift.CycleGAN.models import Generator, CycleGAN
from domain_shift.data_extraction.process_DRIAMS import DRIAMS_bin_to_df

In [12]:
NUMBER_EXECUTIONS = 10

In [13]:
# Load the data
driams_1 = DRIAMS_bin_to_df(settings.DRIAMS_C_PATH)
driams_2 = DRIAMS_bin_to_df(settings.DRIAMS_D_PATH)

In [14]:
# Train Random Forest on DRIAMS 1
X = np.vstack(driams_1["binned_6000"].values)
y = driams_1["species"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

f1_score(y_test, y_pred, average='weighted')

0.936847637188883

In [15]:
# Combine the 'species' columns from both datasets
combined_species = pd.concat([driams_1['species'], driams_2['species']])

# Find the most represented species across both datasets
most_represented_species = combined_species.value_counts().idxmax()
most_represented_species

'Escherichia coli'

In [16]:
# Show the distribution of the most represented species in each dataset
driams_1['species'].value_counts()[most_represented_species], \
driams_2['species'].value_counts()[most_represented_species]

(np.int64(927), np.int64(2013))

In [17]:
# Filter by the 4 species with most representation
filtered_driams_1 = driams_1[driams_1["species"].isin([most_represented_species])]
filtered_driams_2 = driams_2[driams_2["species"].isin([most_represented_species])]

In [18]:
# Test the model on the filtered driams 2
X = np.vstack(filtered_driams_2["binned_6000"].values)
y = filtered_driams_2["species"].values

y_pred = rf.predict(X)

f1_score(y, y_pred, average='weighted')

0.8155339805825242

In [19]:
# Get the data loaders
driams_1_data_loader = get_data_loader(filtered_driams_1)
driams_2_data_loader = get_data_loader(filtered_driams_2)

print("DataLoader 1 length:", len(driams_1_data_loader))
print("DataLoader 2 length:", len(driams_2_data_loader))

DataLoader 1 length: 309
DataLoader 2 length: 671


In [20]:
results = []
# For each execution
for execution in range(NUMBER_EXECUTIONS):
    # Train the CycleGAN
    cycle_gan = CycleGAN()
    cycle_gan.train(driams_1_data_loader, driams_2_data_loader)

    # Generate the synthetic data
    synthetic_driams_2 = []
    for data in driams_2_data_loader:
        output = cycle_gan.generator_2_to_1(data.to("cuda")).cpu().detach()
        synthetic_driams_2.extend(output.cpu().detach().flatten(1))
    synthetic_driams_2 = torch.stack(synthetic_driams_2)

    # Test the model on the synthetic driams 2
    X = np.vstack(synthetic_driams_2.numpy())
    y = filtered_driams_2["species"].values

    y_pred = rf.predict(X)

    f1 = f1_score(y, y_pred, average='weighted')
    results.append(f1)
    print(f"Execution {execution + 1} F1 score: {f1}")

Epoch 1/2, Loss G: 1.5897592306137085, Loss D 1: 0.07869397848844528, Loss D 2: 0.003173904027789831
Epoch 2/2, Loss G: 0.6647424697875977, Loss D 1: 0.2622247636318207, Loss D 2: 0.24312731623649597
Execution 1 F1 score: 0.8258967629046369
Epoch 1/2, Loss G: 1.1421265602111816, Loss D 1: 0.20940235257148743, Loss D 2: 0.160037562251091
Epoch 2/2, Loss G: 0.8310354948043823, Loss D 1: 0.22039151191711426, Loss D 2: 0.15335796773433685
Execution 2 F1 score: 0.8890728476821192
Epoch 1/2, Loss G: 1.3273557424545288, Loss D 1: 0.04698560759425163, Loss D 2: 0.1725698560476303
Epoch 2/2, Loss G: 0.7494907975196838, Loss D 1: 0.21505606174468994, Loss D 2: 0.24596497416496277
Execution 3 F1 score: 0.9154094827586207
Epoch 1/2, Loss G: 1.0256688594818115, Loss D 1: 0.2179175615310669, Loss D 2: 0.14399711787700653
Epoch 2/2, Loss G: 0.7482435703277588, Loss D 1: 0.25088953971862793, Loss D 2: 0.24038715660572052
Execution 4 F1 score: 0.8987964989059081
Epoch 1/2, Loss G: 1.7491519451141357, L

In [21]:
# Print the results
results

[0.8258967629046369,
 0.8890728476821192,
 0.9154094827586207,
 0.8987964989059081,
 0.9115977291159773,
 0.9255404323458767,
 0.9029972752043597,
 0.8990976210008204,
 0.8978921434437449,
 0.9191946308724832]

In [ ]:
# pytorch metric learning tripletes

In [ ]:
# TRY TO multiply the full cycles by values 
# See if cycle is not weighted enough